In [1]:
import pandas as pd
import datetime as dt
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
pesature = pd.read_csv(r'Project\03. Variabilità di Peso nel Confezionamento Caffè\Dataset\Clean\pesature_clean.csv')

In [3]:
# Statistiche cliente
lsl = 245
usl = 255
cpk = 1.33

In [4]:

pesature['sotto_specifica'] = pesature['weight_grams']<lsl
pesature['sopra_specifica'] = pesature['weight_grams']>usl
pesature['target'] = (pesature['weight_grams']>lsl) & (pesature['weight_grams']<usl)

## Analisi per singolo operatore 
Per verificare se la causa delle non conformità è attribuibile al singolo operatore. 
Verifico i turni per singolo operatore e successivamente analizzo le non conformità associate ad ognuno 

In [5]:
# Controllo dei turni assegnati a singolo operatore
operatore_turno = pesature.groupby('operator_id')['shift'].unique()
operatore_turno

operator_id
OP-01         [Notte]
OP-02    [Pomeriggio]
OP-04       [Mattina]
OP-06         [Notte]
OP-07       [Mattina]
OP-09    [Pomeriggio]
OP-11       [Mattina]
OP-15    [Pomeriggio]
Name: shift, dtype: object

In [6]:
# Statistiche pesate per operatore
operatore = pesature.groupby('operator_id').agg(
    turno = ('shift', 'first'),
    campionamenti = ('weight_grams', 'size'),
    media = ('weight_grams', 'mean'),
    mediana = ('weight_grams', 'median'),
    dev_std = ('weight_grams', 'std'),
    sotto = ('sotto_specifica', 'sum'),
    sopra = ('sopra_specifica', 'sum'),
    target = ('target', 'sum')
)

# % di fuori specifica per operatore
operatore['%_out'] = ((operatore['sotto']+operatore['sopra'])/operatore['campionamenti'])*100

operatore



,turno,campionamenti,media,mediana,dev_std,sotto,sopra,target,%_out
operator_id,,,,,,,,,
OP-01,Notte,59,248.771864,248.990,2.142001,4,0,55,6.779661
OP-02,Pomeriggio,49,250.029694,249.935,1.137180,0,0,49,0.000000
OP-04,Mattina,42,250.468333,250.395,2.084024,0,1,41,2.380952
OP-06,Notte,81,249.060123,249.270,2.729855,6,1,74,8.641975
OP-07,Mattina,45,250.131556,250.140,0.975784,0,0,45,0.000000
OP-09,Pomeriggio,50,250.019800,249.935,1.400372,0,0,50,0.000000
OP-11,Mattina,53,250.252830,250.160,1.491264,0,0,53,0.000000
OP-15,Pomeriggio,41,250.072683,249.990,1.176225,0,0,41,0.000000


Entrambi gli operatori del turno di notte hanno una maggior presenza di non conformità, sulla base di questa osservazione escludo che il singolo operatore sia la causa delle non conformità, essendo che entrambi gli operatori non coinvolti.

## Analisi per turno
In seguito all'esclusione degli operatori, proseguo analizzando la distribuzione delle non conformità per singolo turno.

In [7]:

# Statistiche delle pesate per turno
turno = pesature.groupby('shift').agg(
    campionamenti = ('weight_grams', 'size'),
    media = ('weight_grams', 'mean'),
    mediana = ('weight_grams', 'median'),
    dev_std = ('weight_grams', 'std'),
    sotto = ('sotto_specifica', 'sum'),
    sopra = ('sopra_specifica', 'sum'),
    target = ('target', 'sum')
)

# % di fuori specifica per turno
turno['%_out'] = ((turno['sotto']+turno['sopra'])/turno['campionamenti'])*100

# Capability per turno
turno['cpk'] = (np.minimum((turno['media']-lsl)/(3*turno['dev_std']), (usl-turno['media'])/(3*turno['dev_std'])))

turno

,campionamenti,media,mediana,dev_std,sotto,sopra,target,%_out,cpk
shift,,,,,,,,,
Mattina,140,250.278500,250.190,1.559666,0,1,139,0.714286,1.009084
Notte,140,248.938643,249.050,2.494770,10,1,129,7.857143,0.526253
Pomeriggio,140,250.038750,249.935,1.239555,0,0,140,0.000000,1.334148


Il Cpk nei turni di mattina e di notte risulta inferiore alle specifiche del cliente (Mattina = 1.01 e Notte = 0.53).
Nel turno di notte, invece, abbiamo la presenza di valori sotto soglia indicati dal cliente (10 pesature)

In [9]:
from scipy import stats

mattina = pesature[pesature['shift']=='Mattina']['weight_grams']
pomeriggio = pesature[pesature['shift']=='Pomeriggio']['weight_grams']
notte = pesature[pesature['shift']=='Notte']['weight_grams']

f_stat, p_value = stats.f_oneway(mattina, pomeriggio, notte)
print(f_stat, p_value)

21.03471155130388 1.9802861174837233e-09


Il risultato di p-value è molto inferiore a 0.05 questo dimostra che tra i tre turni analizzati è presente una variabilità considerevole.